# Receipt Loader v2
- **Cell 3b** = Pre-process images (auto-rotate landscape photos) — run once per upload
- **Cell 4** = Databricks ai_extract (improved prompt, ~13-14/15 products)
- **Cell 5** = Claude Vision API (best accuracy, ~15/15 products)

Run: 1 → 2 → 3 → **3b** → 4 or 5 → 6 → 7

In [ ]:
%pip install pillow anthropic --quiet

In [ ]:
RECEIPTS_VOLUME = '/Volumes/workspace/superapp/receipts'
TABLE           = 'workspace.superapp.gold_user_purchases'
COMERCIO_ID     = 1
import os
ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY', '')
# ANTHROPIC_API_KEY = dbutils.secrets.get(scope='superapp', key='anthropic_api_key')
print(f'API key set: {bool(ANTHROPIC_API_KEY)}')

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS workspace.superapp.gold_user_purchases (
    ticket_id       STRING        COMMENT 'ID unico del ticket',
    fecha           DATE          COMMENT 'Fecha de la compra',
    id_producto     BIGINT        COMMENT 'FK a silver_prices (matched por barcode)',
    producto        STRING        COMMENT 'Nombre en el ticket',
    precio_pagado   DECIMAL(12,2) COMMENT 'Precio final cobrado',
    precio_base     DECIMAL(12,2) COMMENT 'Precio sin descuento',
    descuento       DECIMAL(12,2) COMMENT 'Monto del descuento (positivo)',
    cantidad        DECIMAL(10,3) COMMENT 'Unidades o kilos',
    unidad          STRING        COMMENT 'un o kg',
    precio_unitario DECIMAL(12,2) COMMENT 'Precio por unidad o por kg',
    barcode_ean13   STRING        COMMENT 'EAN-13',
    id_comercio     INT           COMMENT 'FK a gold_comercios',
    source_file     STRING        COMMENT 'Archivo imagen fuente',
    confidence      DECIMAL(3,2)  COMMENT 'Confianza OCR 0-1',
    created_at      TIMESTAMP
) USING DELTA;

In [ ]:
# Cell 3b — Auto-rotate landscape receipt photos
# Run once after uploading new images to the volume.
# Safe to re-run: skips already-portrait images.

from PIL import Image, ExifTags
import os

def exif_rotation(img):
    """Degrees to rotate from EXIF orientation tag (phones set this)."""
    try:
        exif = img._getexif() or {}
        for tag, val in exif.items():
            if ExifTags.TAGS.get(tag) == 'Orientation':
                return {3: 180, 6: 270, 8: 90}.get(val, 0)
    except Exception:
        pass
    return 0


files = [f for f in os.listdir(RECEIPTS_VOLUME) if f.lower().endswith('.jpg')]
print(f'Found {len(files)} images')

rotated, skipped = [], []
for fname in files:
    fpath = f'{RECEIPTS_VOLUME}/{fname}'
    img = Image.open(fpath)
    orig_w, orig_h = img.width, img.height

    degrees = exif_rotation(img)
    if degrees == 0 and img.width > img.height:
        degrees = 90   # landscape with no EXIF tag — rotate 90

    if degrees:
        img = img.rotate(degrees, expand=True)
        # Save without EXIF so next run skips it
        clean = Image.new(img.mode, img.size)
        clean.paste(img)
        clean.save(fpath, 'JPEG', quality=95)
        rotated.append(fname)
        print(f'  Rotated {degrees}deg: {fname}  ({orig_w}x{orig_h} -> {img.width}x{img.height})')
    else:
        skipped.append(fname)

print(f'
Done: {len(rotated)} rotated, {len(skipped)} already portrait')
print('Ready to run Cell 4 (Databricks) or Cell 5 (Claude API)')

In [ ]:
%sql
-- APPROACH A: Improved Databricks ai_extract
-- Handles COTO: weight items, = prefix, discounts
WITH parsed AS (
  SELECT
    regexp_extract(path, '([^/]+)\\.(jpg|jpeg|png|pdf)$', 1) AS ticket_id,
    path                                                        AS source_file,
    ai_parse_document(content, MAP('version', '2.0'))          AS doc
  FROM READ_FILES('/Volumes/workspace/superapp/receipts/*.jpg', format => 'binaryFile')
),
extracted AS (
  SELECT ticket_id, source_file, doc,
    ai_extract(doc, '["fecha", "comercio", "productos"]',
      MAP('instructions',
        'COTO Argentina supermarket receipt.\n\nFormat rules:\n1. Simple: NOMBRE PRECIO on same line, then BARCODE line, then optional discount line\n2. Weight: PESO x PRECIO_KG line BEFORE product name — precio_final = PESO x PRECIO_KG\n3. = prefix means confirmed final price\n4. Discounts: -XXX,XX line, or N *XX% CLASES line, or 30 OFF PROMO VISA DEBIT line\n5. SKIP: store header, SUBTOTAL, TOTAL, VISA, SU VUELTO, tax lines\n\nReturn JSON with fecha (DD/MM/YYYY), comercio, and productos array:\n[{"producto":str,"cantidad":float,"unidad":"kg|un",\n  "precio_unitario":float,"precio_final":float,\n  "descuento":float (positive, 0 if none),"barcode":str|null}]\n\nCRITICAL: calculate precio_final = cantidad x precio_unitario for weight items.\nExtract ALL products, 100% recall.'
    )
  ) AS result
  FROM parsed
),
products AS (
  SELECT ticket_id, source_file,
    to_date(result:response:fecha::string, 'dd/MM/yyyy') AS fecha,
    explode(from_json(
      to_json(result:response:productos),
      'ARRAY<STRUCT<producto:STRING,cantidad:DOUBLE,unidad:STRING,
                    precio_unitario:DOUBLE,precio_final:DOUBLE,
                    descuento:DOUBLE,barcode:STRING>>'
    )) AS p
  FROM extracted
)
SELECT ticket_id, fecha, p.producto, p.cantidad, p.unidad,
       p.precio_unitario, p.precio_final, p.descuento, p.barcode, source_file
FROM products ORDER BY ticket_id, p.producto;

In [ ]:
# APPROACH B: Claude Vision API
# Requires ANTHROPIC_API_KEY (set in Cell 2)
# Get key at: https://console.anthropic.com/account/keys

import anthropic, base64, json, re, os
from PIL import Image
import io
import pandas as pd

PROMPT = (
    'Extract all products from this Argentine supermarket receipt.\n'
    'Return ONLY a JSON object (no markdown):\n'
    '{"fecha":"DD/MM/YYYY","comercio":"name","ticket_id":"tx number",'
    '"productos":[{"producto":"name","cantidad":1.0,"unidad":"un|kg",'
    '"precio_unitario":0.00,"precio_final":0.00,"descuento":0.00,"barcode":"EAN or null"}]}\n'
    'Rules:\n'
    '- Weight item (e.g. 0.398 x 39999): precio_final = weight x unit_price, unidad=kg\n'
    '- descuento = positive discount amount (0 if none)\n'
    '- barcode = 13-digit EAN-13 if visible\n'
    '- Skip store header, totals, payment lines'
)


def parse_receipt_claude(image_path):
    img = Image.open(image_path)
    if img.width > img.height:
        img = img.rotate(90, expand=True)  # auto-rotate landscape
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=90)
    b64 = base64.b64encode(buf.getvalue()).decode()

    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    resp = client.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=3000,
        messages=[{'role': 'user', 'content': [
            {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/jpeg', 'data': b64}},
            {'type': 'text',  'text': PROMPT}
        ]}]
    )
    text = re.sub(r'```json\s*|```\s*', '', resp.content[0].text.strip())
    return json.loads(text)


def process_all_receipts(volume_path):
    rows = []
    files = [f for f in os.listdir(volume_path) if f.lower().endswith('.jpg')]
    print(f'Found {len(files)} receipt images')
    for fname in files:
        print(f'  Parsing {fname}...', end='')
        try:
            data = parse_receipt_claude(f'{volume_path}/{fname}')
            tid = data.get('ticket_id') or fname.replace('.jpg', '')
            for p in data.get('productos', []):
                rows.append({
                    'ticket_id':       tid,
                    'fecha':           data.get('fecha', ''),
                    'source_file':     fname,
                    'producto':        p.get('producto'),
                    'cantidad':        float(p.get('cantidad') or 1),
                    'unidad':          p.get('unidad', 'un'),
                    'precio_unitario': p.get('precio_unitario'),
                    'precio_final':    p.get('precio_final'),
                    'descuento':       float(p.get('descuento') or 0),
                    'barcode_ean13':   p.get('barcode'),
                    'confidence':      0.97,
                })
            print(f' {len(data.get("productos", []))} products OK')
        except Exception as e:
            print(f' ERROR: {e}')
    return pd.DataFrame(rows)


if not ANTHROPIC_API_KEY:
    print('Set ANTHROPIC_API_KEY in Cell 2.')
    print('Get your key at: https://console.anthropic.com/account/keys')
else:
    receipts_df = process_all_receipts(RECEIPTS_VOLUME)
    print(f'Total rows: {len(receipts_df)}')
    display(receipts_df)

In [ ]:
# Save to gold — works after Cell 4 (create temp view) or Cell 5 (DataFrame in memory)
# After Cell 4: add CREATE OR REPLACE TEMP VIEW receipt_extract AS <cell4 query>
# After Cell 5: receipts_df is already loaded

if 'receipts_df' in dir():
    spark.createDataFrame(receipts_df).createOrReplaceTempView('receipt_extract')
    print(f'Loaded {len(receipts_df)} rows into temp view receipt_extract')

spark.sql(f"""
INSERT INTO {TABLE}
WITH matched AS (
  SELECT
    r.ticket_id,
    to_date(r.fecha, 'dd/MM/yyyy')  AS fecha,
    r.producto, r.cantidad, r.unidad, r.precio_unitario,
    r.precio_final                  AS precio_pagado,
    r.precio_final                  AS precio_base,
    r.descuento, r.barcode_ean13,
    sp.id_producto,
    {COMERCIO_ID}                   AS id_comercio,
    r.source_file,
    COALESCE(CAST(r.confidence AS DECIMAL(3,2)), 0.90) AS confidence,
    current_timestamp()             AS created_at
  FROM receipt_extract r
  LEFT JOIN workspace.superapp.silver_prices sp
    ON  sp.productos_ean = r.barcode_ean13
    AND r.barcode_ean13 IS NOT NULL
)
SELECT ticket_id, fecha, id_producto, producto, precio_pagado, precio_base,
       descuento, cantidad, unidad, precio_unitario, barcode_ean13,
       id_comercio, source_file, confidence, created_at
FROM matched
WHERE fecha IS NOT NULL AND producto IS NOT NULL AND precio_pagado IS NOT NULL
""")
print('Saved.')

In [ ]:
%sql
SELECT
  fecha, ticket_id,
  COUNT(*)                                                                     AS items,
  ROUND(SUM(precio_pagado), 2)                                                AS total_pagado,
  ROUND(SUM(descuento), 2)                                                    AS total_ahorro,
  ROUND(SUM(descuento)/NULLIF(SUM(precio_pagado+descuento),0)*100, 1)         AS pct_ahorro,
  COUNT(CASE WHEN id_producto IS NOT NULL THEN 1 END)                         AS matched_sepa,
  source_file
FROM workspace.superapp.gold_user_purchases
GROUP BY fecha, ticket_id, source_file
ORDER BY fecha DESC;